# 05a — Patient-cluster sensitivity

## tl;dr

Compare ordinary and patient-cluster confidence intervals on identical complete-case admissions in the refined primary and exploratory first-day-SOFA core models. The point estimate is shared; inference changes. The default run is a synthetic demonstration, not a MIMIC result.

## Context & Methods

Keep the current exposure, outcome, age, sex, era strata, covariates and Efron ties. Fit unpenalized Cox models and sum Efron score residuals over all admissions of each patient, including across era strata. Form an HC0 sandwich covariance with asymptotic normal Wald intervals and p-values.

### Key assumptions
- Patients are independent clusters; admissions within a patient may be dependent. This does not fit a frailty or recurrent-event process model.
- No weights, delayed entry, or finite-cluster correction are used.
- The core model retains the published first-day-SOFA timing limitation.
- Tied event/censor times are handled deterministically; synthetic coefficients and covariance are checked against R `survival::coxph`.
- Current-data execution must reproduce the ordinary saved reference before returning a result.

## Data

Run from the repository root or `notebooks/` after `uv sync --locked`. The existing bounded BigQuery loader is used only for explicit `current` mode. Pseudonymous keys stay in memory; never print source rows. Use `local` mode to inspect a previously verified aggregate run without another query.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from survival_sensitivity import run_supplement, load_local_bundle, plot_supplement
%matplotlib inline
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})

In [ ]:
# Default: artificial data only. Never display a patient-level DataFrame.
DATA_MODE = "synthetic"  # "synthetic", "current", or "local"
LOCAL_RESULT = REPO_ROOT / "reports/local_supplements/sensitivity_run02/aggregate_results.json"
BOOTSTRAP_REPLICATES = 1000 if DATA_MODE == "current" else 200

if DATA_MODE == "local":
    bundle = load_local_bundle(LOCAL_RESULT, REPO_ROOT, "cluster")
elif DATA_MODE in {"synthetic", "current"}:
    bundle = run_supplement("cluster", repo=REPO_ROOT, clinical=DATA_MODE == "current",
                            bootstrap_replicates=BOOTSTRAP_REPLICATES)
else:
    raise ValueError("Unknown DATA_MODE")
result = bundle["results"]["cluster"]
print({"data_origin": bundle["data_origin"], "supplement_version": bundle["supplement_version"],
       "main_analysis_updated": bundle["main_analysis_updated"]})

## Results

The data-origin label above controls interpretation. Synthetic outputs validate execution only.

### Repeated admissions and ties

In [ ]:
display(pd.DataFrame([{k: m[k] for k in ["n", "events", "patients", "patients_with_multiple_admissions", "admissions_from_repeat_patients", "tied_event_sets_within_era"]}
                      for m in result["models"].values()], index=list(result["models"])))

### Estimates on the same admissions

In [ ]:
rows = []
for name, model in result["models"].items():
    e = model["estimates"]["raas_pre_icu"]
    rows.append({"Model": name, "Admissions": model["n"], "Patients": model["patients"],
                 "RAAS HR": e["hr"],
                 "Ordinary 95% CI": f"{e['ordinary']['lower_95']:.3f}–{e['ordinary']['upper_95']:.3f}",
                 "Cluster 95% CI": f"{e['patient_cluster']['lower_95']:.3f}–{e['patient_cluster']['upper_95']:.3f}",
                 "Ordinary p": e["ordinary"]["p"], "Cluster p": e["patient_cluster"]["p"]})
display(pd.DataFrame(rows).style.format({"RAAS HR": "{:.3f}", "Ordinary p": "{:.4g}", "Cluster p": "{:.4g}"}))

### Visual comparison

In [ ]:
ANALYSIS = "cluster"
figure = plot_supplement(bundle, ANALYSIS)
display(figure)
plt.close(figure)

## Takeaways

The estimate table and forest plot compare uncertainty methods, not different fitted exposure effects. Review the primary cluster interval alongside the ordinary interval regardless of significance. Changing the published main-analysis inference requires updating its provenance and independent verification together; this notebook does not overwrite them.

See [the supplement guide](../docs/SURVIVAL_SENSITIVITY_SUPPLEMENTS.md) for the execution commands, data handling, reference checks and limitations. Tracked notebook outputs remain stripped; keep executed clinical copies under ignored `reports/local_supplements/`.